# ⚙️ Data Preparation for Route Optimization

**Author:** Miguel Vásquez  
**Date:** October 2025  
**Role:** Data Engineer | Machine Learning Engineer  

---

## 1. Introduction
In this notebook, we prepare the Solomon VRPTW benchmark data for route optimization tasks.
Our goal is to transform the raw logistics data into structured, machine-readable formats suitable for algorithmic optimization (e.g., with OR-Tools).

The preparation process ensures consistency across multiple instances (C, R, RC types) and creates intermediary outputs such as **distance matrices** and **time constraints**, which are essential for solving Vehicle Routing Problems with Time Windows (VRPTW).

---

## 2. Objectives
1. Clean and standardize column names, data types, and coordinate formats.
2. Simulate and structure delivery request data.
3. Generate and validate distance matrices between all customer nodes.
4. Define service constraints: capacity, service time, and time windows.
5. Save processed datasets for downstream use in optimization algorithms.

---

## 3. Input Data
We will use the unified dataset created in `01_eda.ipynb`, which aggregates all Solomon VRPTW instances into a single DataFrame (`vrptw_df`).
This allows us to build flexible pipelines that can process any instance or scenario type (C, R, RC).

---

## 4. Setup & Data Loading
In this section, we load the cleaned and standardized Solomon VRPTW dataset generated from the previous notebook (`01_data_extraction_eda.ipynb`).
This ensures data consistency across all stages of the project and allows a modular workflow, where each step builds upon reproducible artifacts.
The dataset will be used to perform preprocessing, feature engineering, and the creation of data structures required for optimization models.

In [1]:
import sys
from pathlib import Path

# Add the parent directory to sys.path so 'src' can be imported
sys.path.append(str(Path.cwd().parent))

from src.data_utils import ensure_project_root

# Ensure we're in the project root
ensure_project_root()

from src.data_utils import load_processed_data
cleaned_path = "data/processed/vrptw_cleaned.csv"
data = load_processed_data(cleaned_path)

Changed working directory to project root: `c:\Users\Miguel\portfolio\Route-Optimization-VRPTW`

Loading processed dataset from: data/processed/vrptw_cleaned.csv
✅ Loaded dataset with 5656 rows and 9 columns.


## 5. Feature Preparation & Normalization
Before moving into optimization, we prepare the features to ensure that spatial and temporal values are comparable.
Although the Solomon dataset is already well-structured, normalizing features helps solvers (e.g., OR-Tools, heuristics, or metaheuristics) converge faster and behave more stably across scenarios.

### Why normalization matters
Vehicle Routing optimization involves multiple scales of data:
- **Spatial** coordinates (`XCOORD`, `YCOORD`) may vary across hundreds of units.
- **Temporal** windows (`READYTIME`, `DUEDATE`) may span thousands of minutes.
- **Demand** (`DEMAND`) depends on instance type and capacity constraints.
By scaling all of them into comparable ranges, we prevent solvers from overemphasizing any single feature.

---

### 5.1 Handle missing values
Although Solomon VRPTW data is typically clean, we perform a consistency check to detect any potential missing values from parsing or concatenation.

In [2]:
from IPython.display import Markdown, display

missing_summary = data.isnull().sum()
display(Markdown("Missing values summary:"))
display(missing_summary[missing_summary > 0])

# If any missing values exist, handle them
vrptw_df = data.fillna({
    "DEMAND": 0,
    "SERVICETIME": data["SERVICE_TIME"].median(),
    "READYTIME": 0,
    "DUEDATE": data["DUE_DATE"].max()
})

display(Markdown("✅ Missing values handled successfully."))

Missing values summary:

Series([], dtype: int64)

✅ Missing values handled successfully.

### 5.2 Pre-Normalization Check Data

Before encoding or scaling features, we inspect data types and ensure all numeric columns are correctly typed.  
This step helps prevent downstream errors during normalization (e.g., dividing by a string column).

We’ll check column types, detect inconsistencies, and fix any misformatted values (such as `READY_TIME`).

In [3]:
import pandas as pd
display(Markdown("🔍 Data type inspection before normalization:"))
vrptw_df.info()

🔍 Data type inspection before normalization:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5656 entries, 0 to 5655
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   CUST_NO       5656 non-null   int64 
 1   XCOORD        5656 non-null   int64 
 2   YCOORD        5656 non-null   int64 
 3   DEMAND        5656 non-null   int64 
 4   READY_TIME    5656 non-null   object
 5   DUE_DATE      5656 non-null   int64 
 6   SERVICE_TIME  5656 non-null   int64 
 7   INSTANCE      5656 non-null   object
 8   TYPE          5656 non-null   object
dtypes: int64(6), object(3)
memory usage: 397.8+ KB


In [4]:
# Fix potential type issues
if vrptw_df["READY_TIME"].dtype == "object":
    vrptw_df["READY_TIME"] = pd.to_numeric(vrptw_df["READY_TIME"], errors="coerce")
    n_na = vrptw_df["READY_TIME"].isna().sum()
    if n_na > 0:
        print(f"{n_na} invalid READY_TIME values replaced with 0.")
        vrptw_df["READY_TIME"] = vrptw_df["READY_TIME"].fillna(0)

display(Markdown("Column types after correction:"))
display(vrptw_df.dtypes)

1 invalid READY_TIME values replaced with 0.


Column types after correction:

CUST_NO           int64
XCOORD            int64
YCOORD            int64
DEMAND            int64
READY_TIME      float64
DUE_DATE          int64
SERVICE_TIME      int64
INSTANCE         object
TYPE             object
dtype: object

### 5.3 Encode categorical variables
The Solomon dataset includes an instance **TYPE** (C, R, or RC), representing customer distribution and time window characteristics.
For optimization models, we encode this as numeric to allow solvers to leverage instance-level patterns if needed.

In [5]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
vrptw_df["TYPE_ENCODED"] = encoder.fit_transform(vrptw_df["TYPE"])

display(Markdown("Encoded instance types:"))
display(vrptw_df[["TYPE", "TYPE_ENCODED"]].drop_duplicates().sort_values("TYPE"))

Encoded instance types:

,TYPE,TYPE_ENCODED
0,C,0
1717,R,1
4040,RC,2


### 5.4  Feature Normalization for VRPTW Optimization
To align all input dimensions, we apply **min-max scaling** to spatial and temporal variables, and **max scaling** to demand

In [6]:
df = vrptw_df.copy()

# --- Normalize coordinates (0–1 range) ---
df["XCOORD"] = (df["XCOORD"] - df["XCOORD"].min()) / (df["XCOORD"].max() - df["XCOORD"].min())
df["YCOORD"] = (df["YCOORD"] - df["YCOORD"].min()) / (df["YCOORD"].max() - df["YCOORD"].min())

# --- Normalize time columns by total time horizon ---
time_horizon = df["DUE_DATE"].max()
df["READY_TIME"] = df["READY_TIME"] / time_horizon
df["DUE_DATE"] = df["DUE_DATE"] / time_horizon
df["SERVICE_TIME"] = df["SERVICE_TIME"] / time_horizon

# --- Normalize demand by maximum observed ---
df["DEMAND"] = df["DEMAND"] / df["DEMAND"].max()

display(Markdown("✅ Features normalized for optimization scale."))
display(df.head())

✅ Features normalized for optimization scale.

,CUST_NO,XCOORD,YCOORD,DEMAND,READY_TIME,DUE_DATE,SERVICE_TIME,INSTANCE,TYPE,TYPE_ENCODED
0,1,0.421053,0.573171,0.0,0.000000,0.364602,0.000000,C101,C,0
1,2,0.473684,0.792683,0.2,0.269027,0.285251,0.026549,C101,C,0
2,3,0.473684,0.817073,0.6,0.243363,0.256637,0.026549,C101,C,0
3,4,0.442105,0.768293,0.2,0.019174,0.043068,0.026549,C101,C,0
4,5,0.442105,0.792683,0.2,0.214454,0.230678,0.026549,C101,C,0


## 6. Save Processed Dataset

After cleaning, validating, and normalizing the Solomon VRPTW dataset,  
we now save the final processed version for downstream optimization tasks.  

This dataset will serve as the standardized input for:
- Route optimization models (e.g., OR-Tools, heuristics, or metaheuristics),
- Performance benchmarking across different instance types (C, R, RC),
- Future deployment or visualization pipelines.

By saving a consistent intermediate artifact, we ensure **reproducibility**  
and can skip previous preprocessing stages when iterating on optimization logic.

In [7]:
from src.data_utils import save_processed_data

# Save the normalized dataset ready for optimization
output_path = save_processed_data(df, "vrptw_ready_for_optimization.csv")

✅ Data saved successfully at: data/processed\vrptw_ready_for_optimization.csv
Rows: 5656 | Columns: 10


## 7. Summary & Next Steps

**In this notebook, we accomplished:**
1. Verified and loaded the unified Solomon VRPTW dataset.  
2. Checked data integrity and types before processing.  
3. Encoded categorical variables for scenario types (C, R, RC).  
4. Normalized spatial, temporal, and demand-related features.  
5. Saved the fully prepared dataset for optimization experiments.

---

### Next Steps
In the following notebook, we will begin building the **optimization engine**,  
starting with:
- Constructing **distance and time matrices** from coordinates,  
- Defining **vehicle and time window constraints**,  
- Implementing a **baseline solver** using Google OR-Tools.

This marks the transition from *data preparation* 🧹 to *route optimization modeling* 🚚.